# 🎬 BENY-JOE IA — Génération Vidéo & Image HD
**Fondé par KHEDIM BENYAKHLEF dit BENY-JOE**

| Feature | Détail |
|---|---|
| 📐 Résolution | **1024×576 HD 16:9** |
| 🎬 Vidéo | **SD 1.5 multi-frames (CPU/TPU)** |
| 🎤 Voix OFF | **gTTS FR/EN/AR** |
| 🎵 Musique | **MusicGen-small IA** |
| ⚡ Device | **TPU v5e Kaggle (JAX + PyTorch CPU)** |
| 🌐 Plateforme | **BENY-JOE IA sur Render** |

---
## ▶️ ORDRE D'EXÉCUTION :
1. **Cellule 0** — Vérification TPU/device
2. **Cellule 1** — Secrets Kaggle (tokens)
3. **Cellule 2** — Installation des dépendances
4. **Cellule 3** — Init device + Chargement des modèles ✅ CORRIGÉ
5. **Cellule 4** — Serveur Flask + Tunnel ngrok
6. **Cellule 5** — Watcher vidéos (auto-push GitHub → Render)
7. **Cellule 6** — Keep-alive permanent ♾️

---
> ⚠️ **TPU v5e Kaggle** : JAX est natif. PyTorch tourne sur **CPU** (pas CUDA sur TPU v5e).
> Ne jamais faire `.to('cuda')` — utilisez `.to('cpu')` ou `torch_device` auto-détecté.
> Sessions Kaggle : max ~9h. Le keep-alive (cellule 6) maintient la session active.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 — Vérification device (TPU / GPU / CPU)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

import subprocess, sys

print('🔍 Vérification environnement Kaggle...')
print()

# 1. JAX natif (TPU v5e)
try:
    import jax
    import jax.numpy as jnp
    devices = jax.devices()
    print(f'✅ JAX {jax.__version__} — {len(devices)} device(s) : {devices}')
    x = jnp.ones((256, 256))
    y = jnp.dot(x, x)
    print(f'✅ Calcul JAX OK — shape : {y.shape}')
    JAX_OK = True
except Exception as e:
    print(f'⚠️  JAX : {e}')
    JAX_OK = False

print()

# 2. PyTorch — détection device réelle
try:
    import torch
    print(f'✅ PyTorch {torch.__version__}')
    
    # Sur TPU v5e Kaggle : CUDA n'existe PAS → on utilise CPU
    if torch.cuda.is_available():
        TORCH_DEVICE = 'cuda'
        print(f'✅ CUDA disponible — GPU : {torch.cuda.get_device_name(0)}')
    else:
        TORCH_DEVICE = 'cpu'
        print('ℹ️  CUDA absent — PyTorch tournera sur CPU (normal sur TPU v5e)')
        print('   → Les modèles utilisent torch.float32 sur CPU')
    
    PYTORCH_OK = True
except Exception as e:
    print(f'⚠️  PyTorch : {e}')
    TORCH_DEVICE = 'cpu'
    PYTORCH_OK = False

print()
print(f'📋 Résumé :')
print(f'   JAX     : {"✅" if JAX_OK else "❌"}')
print(f'   PyTorch : {"✅" if PYTORCH_OK else "❌"}')
print(f'   Device  : {TORCH_DEVICE.upper()}')
print()
print('👉 Si JAX et PyTorch sont OK → passez à la Cellule 1')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Chargement des secrets Kaggle                         ║
# ║  Prérequis : Kaggle → Settings → Secrets → Add secrets :           ║
# ║    GITHUB_TOKEN  (ex: ghp_xxxx)                                    ║
# ║    NGROK_TOKEN   (ex: 2abc_xxxx)                                   ║
# ║    GITHUB_REPO   (ex: username/benyjoe-videos)                     ║
# ║    BENYJOE_SECRET (ex: benyjoe-secret-2025)                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os

GITHUB_TOKEN  = ''
NGROK_TOKEN   = ''
GITHUB_REPO   = ''
BENYJOE_SECRET = ''

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    GITHUB_TOKEN   = _secrets.get_secret('GITHUB_TOKEN')
    NGROK_TOKEN    = _secrets.get_secret('NGROK_TOKEN')
    # Optionnels — ne crashent pas si absents
    try: GITHUB_REPO    = _secrets.get_secret('GITHUB_REPO')
    except: pass
    try: BENYJOE_SECRET = _secrets.get_secret('BENYJOE_SECRET')
    except: pass
    print('✅ Tokens chargés depuis Kaggle Secrets !')
except Exception as e:
    print(f'⚠️  Kaggle Secrets : {e}')
    GITHUB_TOKEN   = os.environ.get('GITHUB_TOKEN', '')
    NGROK_TOKEN    = os.environ.get('NGROK_TOKEN', '')
    GITHUB_REPO    = os.environ.get('GITHUB_REPO', '')
    BENYJOE_SECRET = os.environ.get('BENYJOE_SECRET', '')

# Valeur par défaut secret
if not BENYJOE_SECRET:
    BENYJOE_SECRET = 'benyjoe-secret-2025'

# Exporter dans l'environnement
os.environ['GITHUB_TOKEN']   = GITHUB_TOKEN
os.environ['NGROK_TOKEN']    = NGROK_TOKEN
os.environ['GITHUB_REPO']    = GITHUB_REPO
os.environ['BENYJOE_SECRET'] = BENYJOE_SECRET
os.environ['RENDER_URL']     = 'https://benyjoe-ia.onrender.com'
os.environ['DEVICE_NAME']    = 'Kaggle-TPU-v5e'

print()
print(f'  GITHUB_TOKEN   : {"✅ présent" if GITHUB_TOKEN else "❌ absent — OBLIGATOIRE pour push vidéos"}')
print(f'  NGROK_TOKEN    : {"✅ présent" if NGROK_TOKEN else "❌ absent — OBLIGATOIRE pour tunnel"}')
print(f'  GITHUB_REPO    : {GITHUB_REPO if GITHUB_REPO else "⚠️  absent (format: username/repo)"}')
print(f'  BENYJOE_SECRET : {"✅ présent" if BENYJOE_SECRET else "⚠️  défaut utilisé"}')
print()
print('👉 Passez à la Cellule 2')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Installation des dépendances (TPU v5e Kaggle)         ║
# ║  ✅ UNE SEULE installation propre — pas de réinstallations         ║
# ║  ⚠️  NE PAS réinstaller torch ou torch_xla                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

import subprocess, sys

def pip_install(pkg, extra=''):
    cmd = f'{sys.executable} -m pip install -q {extra} {pkg}'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=900)
    status = '✅' if r.returncode == 0 else '⚠️'
    short = pkg.split()[0][:50]
    print(f'  {status} {short}')
    if r.returncode != 0 and r.stderr:
        last_err = [l for l in r.stderr.strip().splitlines() if l.strip()][-1:]
        if last_err:
            print(f'     └─ {last_err[0][:120]}')
    return r.returncode == 0

print('╔══════════════════════════════════════════════╗')
print('║  BENY-JOE IA — Installation (Kaggle TPU)    ║')
print('╚══════════════════════════════════════════════╝')
print()

# ── 0. Supprimer tensorflow (conflit avec torch-xla) ──────────────────
print('🔧 Nettoyage conflits...')
subprocess.run('pip uninstall -y tensorflow tensorflow-cpu 2>/dev/null',
               shell=True, capture_output=True)
print('  ✅ tensorflow supprimé')
print()

# ── 1. Versions FIXES et COMPATIBLES TPU v5e ──────────────────────────
# numpy 1.26.4 = compatible PyTorch 2.x + diffusers + audiocraft
print('🔧 NumPy (version stable)...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'numpy==1.26.4', '--force-reinstall'],
    check=False, capture_output=True
)
print('  ✅ NumPy 1.26.4')
print()

# ── 2. Diffusion (versions FIXES compatibles) ─────────────────────────
print('🔧 Diffusers / Transformers / Accelerate...')
# diffusers 0.24.0 = AnimateDiffPipeline stable
# transformers 4.38.2 = compatible avec diffusers 0.24.0
for p in [
    'diffusers==0.24.0',
    'transformers==4.38.2',
    'accelerate==0.30.1',
    'safetensors==0.4.3',
    'omegaconf',
    'einops',
    'huggingface_hub==0.23.2',
    'compel',
]:
    pip_install(p)
print()

# ── 3. Vidéo & Image ──────────────────────────────────────────────────
print('🔧 Vidéo & Image...')
for p in [
    'imageio>=2.34.1',
    'imageio-ffmpeg',
    'opencv-python-headless',
    'Pillow>=10.3.0',
    'moviepy==1.0.3',
    'ffmpeg-python',
]:
    pip_install(p)
print()

# ── 4. Audio ──────────────────────────────────────────────────────────
print('🔧 Audio (gTTS + MusicGen)...')
for p in ['pydub', 'scipy', 'soundfile', 'gtts']:
    pip_install(p)

print('  🎵 Audiocraft / MusicGen...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/facebookresearch/audiocraft.git'],
    capture_output=True, text=True, timeout=600
)
print(f'  {"✅" if r.returncode == 0 else "⚠️"} audiocraft')
print()

# ── 5. Serveur & Tunnel ───────────────────────────────────────────────
print('🔧 Flask + Ngrok...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'blinker==1.7.0', '--ignore-installed'],
    check=False, capture_output=True
)
for p in ['flask', 'flask-cors', 'pyngrok>=7.0.0', 'requests', 'python-dotenv']:
    pip_install(p)
print()

# ── 6. Vérification finale ────────────────────────────────────────────
print('🔧 Vérification finale...')
import importlib

for mod, name in [('numpy', 'NumPy'), ('diffusers', 'Diffusers'),
                  ('transformers', 'Transformers'), ('flask', 'Flask')]:
    try:
        m = importlib.import_module(mod)
        print(f'  ✅ {name} : {m.__version__}')
    except:
        print(f'  ❌ {name} non disponible')

# Test import AnimateDiff
try:
    from diffusers import AnimateDiffPipeline
    print('  ✅ AnimateDiffPipeline importable')
except Exception as e:
    print(f'  ⚠️  AnimateDiff : {e} — vidéo multi-frames utilisera SD 1.5 simple')

print()
print('✅ Installation terminée — passez à la Cellule 3')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Init JAX/TPU + FlaxSD ✅ VERSION CORRIGÉE            ║
# ║  Modèle : CompVis/stable-diffusion-v1-4 revision=bf16             ║
# ║  8 chips TPU v3 en parallèle → ~2-5s par image                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, gc, time, warnings
import jax
import jax.numpy as jnp
import numpy as np
warnings.filterwarnings('ignore')

OUTPUTS_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print('╔══════════════════════════════════════════════════════╗')
print('║  BENY-JOE IA — Init JAX/TPU v3-8                   ║')
print('╚══════════════════════════════════════════════════════╝')
print()

# ── Vérification TPU ──────────────────────────────────────────────────
devices   = jax.devices()
N_DEVICES = len(devices)
print(f'✅ JAX {jax.__version__}')
print(f'✅ {N_DEVICES} chip(s) : {devices[0].device_kind}')
print()

# Variables globales
pipe_flax    = None
p_params     = None
pipe_image   = None
pipe_video   = None
music_model  = None
IMAGE_OK     = False
VIDEO_OK     = False
MUSIC_OK     = False
ANIMATEDIFF  = False
TORCH_DEVICE = 'cpu'
JAX_MODE     = False

# ── 1. FlaxStableDiffusion JAX (modèle correct) ───────────────────────
print('🎨 Chargement FlaxSD (CompVis/stable-diffusion-v1-4 bf16)...')
print('   ⏳ Téléchargement + compilation JIT (~3-5 min première fois)')
print()

# Essayer plusieurs combinaisons modèle/revision
flax_configs = [
    ('CompVis/stable-diffusion-v1-4', 'bf16',  jnp.bfloat16),
    ('CompVis/stable-diffusion-v1-4', 'flax',  jnp.bfloat16),
    ('runwayml/stable-diffusion-v1-5', 'flax', jnp.bfloat16),
    ('runwayml/stable-diffusion-v1-5', None,   jnp.float32),
]

for model_id, revision, dtype in flax_configs:
    try:
        from diffusers import FlaxStableDiffusionPipeline
        from flax.jax_utils import replicate

        kwargs = dict(dtype=dtype, safety_checker=None,
                      requires_safety_checker=False)
        if revision:
            kwargs['revision'] = revision

        print(f'  → Essai : {model_id} revision={revision}...')
        pipe_flax, params = FlaxStableDiffusionPipeline.from_pretrained(
            model_id, **kwargs
        )
        p_params  = replicate(params)
        JAX_MODE  = True
        IMAGE_OK  = True
        print(f'  ✅ FlaxSD chargé ! ({model_id} / {revision})')
        print(f'  ✅ Répliqué sur {N_DEVICES} chips TPU')
        break
    except Exception as e:
        print(f'  ⚠️  {model_id}/{revision} : {str(e)[:80]}')
        continue

# ── 2. Fallback PyTorch CPU si Flax échoue ───────────────────────────
if not JAX_MODE:
    print()
    print('⚠️  FlaxSD non disponible — fallback PyTorch CPU')
    try:
        import torch
        from diffusers import StableDiffusionPipeline
        pipe_image = StableDiffusionPipeline.from_pretrained(
            'runwayml/stable-diffusion-v1-5',
            torch_dtype=torch.float32,
            safety_checker=None,
        ).to('cpu')
        pipe_image.enable_attention_slicing()
        IMAGE_OK = True
        print('✅ SD 1.5 PyTorch CPU chargé (fallback)')
    except Exception as e:
        print(f'❌ Fallback : {e}')

# ── 3. Fonctions de génération ────────────────────────────────────────
def generate_image_jax(prompt, negative_prompt='low quality, blurry',
                        height=512, width=512, steps=20,
                        guidance=7.5, seed=0):
    from flax.jax_utils import replicate
    from flax.training.common_utils import shard
    prompt_ids = pipe_flax.prepare_inputs([prompt] * N_DEVICES)
    neg_ids    = pipe_flax.prepare_inputs([negative_prompt] * N_DEVICES)
    rng = jax.random.split(jax.random.PRNGKey(seed), N_DEVICES)
    output, _ = pipe_flax(
        prompt_ids, p_params, rng,
        neg_prompt_ids=neg_ids,
        num_inference_steps=steps,
        guidance_scale=guidance,
        height=height, width=width,
        jit=True,
    )
    imgs = output.reshape((N_DEVICES,) + output.shape[-3:])
    return pipe_flax.numpy_to_pil(imgs)

def generate_image_cpu(prompt, negative_prompt='low quality, blurry',
                        height=512, width=512, steps=25,
                        guidance=7.5, seed=0):
    import torch
    g = torch.Generator(device='cpu').manual_seed(seed)
    result = pipe_image(
        prompt=prompt, negative_prompt=negative_prompt,
        height=height, width=width,
        num_inference_steps=steps, guidance_scale=guidance, generator=g,
    )
    return result.images

def generate_image(prompt, negative_prompt='low quality, blurry, distorted',
                   height=512, width=512, steps=20, guidance=7.5, seed=0):
    if JAX_MODE and pipe_flax is not None:
        return generate_image_jax(prompt, negative_prompt,
                                   height, width, steps, guidance, seed)
    elif pipe_image is not None:
        return generate_image_cpu(prompt, negative_prompt,
                                   height, width, steps, guidance, seed)
    raise RuntimeError('Aucun modèle disponible')

# ── 4. Test JIT si JAX actif ──────────────────────────────────────────
if JAX_MODE:
    print()
    print('⚙️  Test compilation JIT (peut prendre 2-5 min)...')
    t0 = time.time()
    try:
        test_imgs = generate_image_jax(
            'beautiful landscape', steps=4, seed=0)
        elapsed = time.time() - t0
        print(f'✅ JIT compilé en {elapsed:.0f}s')
        print(f'   {len(test_imgs)} images générées (une par chip)')
        test_imgs[0].save(os.path.join(OUTPUTS_DIR, 'test_jax.png'))
        VIDEO_OK = True
    except Exception as e:
        print(f'⚠️  JIT : {e}')
        JAX_MODE = False
        VIDEO_OK = IMAGE_OK

# ── 5. MusicGen ──────────────────────────────────────────────────────
print()
print('🎵 Chargement MusicGen-small...')
try:
    from audiocraft.models import MusicGen
    music_model = MusicGen.get_pretrained('small')
    MUSIC_OK = True
    print('✅ MusicGen chargé !')
except Exception as e:
    print(f'⚠️  MusicGen absent : {e}')

VIDEO_OK = IMAGE_OK

# ── Résumé ────────────────────────────────────────────────────────────
mode = f"JAX/TPU ({N_DEVICES} chips)" if JAX_MODE else "PyTorch CPU"
print()
print('╔══════════════════════════════════════════════════════╗')
print(f'║  🎨 Image  : {"✅ PRÊT" if IMAGE_OK else "❌ ÉCHEC":<28}║')
print(f'║  🎬 Vidéo  : {"✅ PRÊT" if VIDEO_OK else "❌ ÉCHEC":<28}║')
print(f'║  🎵 Musique: {"✅ PRÊT" if MUSIC_OK else "⚠️  silencieux":<28}║')
print(f'║  ⚡ Mode   : {mode:<28}║')
print('╚══════════════════════════════════════════════════════╝')
print()
if JAX_MODE:
    print('⚡ VITESSE TPU :')
    print(f'   1 image  → ~2-5s  ({N_DEVICES} images en parallèle)')
    print(f'   1 clip   → ~15-30s')
    print(f'   1min vid → ~5-10min')
else:
    print('⚠️  Mode CPU actif — vitesse lente')
    print('   → Vérifiez : Kaggle → Settings → Accelerator → TPU v3-8')
print()
print('👉 Passez à la Cellule 4')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Serveur Flask + Tunnel ngrok ✅ CORRIGÉ               ║
# ║  ⚠️  Cette cellule tourne en ARRIÈRE-PLAN (thread)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, gc, time, uuid, json, logging, threading, io
import numpy as np
import torch
from pathlib import Path
from datetime import datetime, timezone
from flask import Flask, request, jsonify, send_file, abort
from flask_cors import CORS
from gtts import gTTS
from pydub import AudioSegment

OUTPUTS_DIR   = '/kaggle/working/outputs'
FLASK_PORT    = 8765
RENDER_URL    = os.environ.get('RENDER_URL',     'https://benyjoe-ia.onrender.com')
SECRET_KEY    = os.environ.get('BENYJOE_SECRET', 'benyjoe-secret-2025')
DEVICE_NAME   = os.environ.get('DEVICE_NAME',   'Kaggle-TPU-v5e')
Path(OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)

log = logging.getLogger('BENYJOE')
logging.basicConfig(level=logging.INFO)

# ── Jobs store ────────────────────────────────────────────────────────
jobs = {}
lock = threading.Lock()

def set_job(jid, **kw):
    with lock:
        if jid not in jobs:
            jobs[jid] = {'id': jid, 'status': 'pending', 'progress': 0,
                         'step': '', 'result': None, 'error': None}
        jobs[jid].update(kw)

# ── Helpers audio ─────────────────────────────────────────────────────
def generate_voice(text, lang='fr', out_path=None):
    if out_path is None:
        out_path = os.path.join(OUTPUTS_DIR, f'voice_{uuid.uuid4().hex[:8]}.mp3')
    try:
        tts = gTTS(text=text[:500], lang=lang, slow=False)
        tts.save(out_path)
        return out_path
    except Exception as e:
        log.warning(f'gTTS : {e}')
        return None

def generate_music(prompt, duration=10, out_path=None):
    if out_path is None:
        out_path = os.path.join(OUTPUTS_DIR, f'music_{uuid.uuid4().hex[:8]}.wav')
    try:
        if music_model is None:
            return None
        music_model.set_generation_params(duration=min(duration, 30))
        wav = music_model.generate([prompt])
        import soundfile as sf
        audio_np = wav[0].cpu().numpy()
        if audio_np.ndim == 2:
            audio_np = audio_np[0]
        sf.write(out_path, audio_np, samplerate=32000)
        return out_path
    except Exception as e:
        log.warning(f'MusicGen : {e}')
        return None

def mix_audio_with_video(video_path, voice_path, music_path, out_path):
    try:
        from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip
        clip = VideoFileClip(video_path)
        audio_clips = []
        if voice_path and os.path.exists(voice_path):
            audio_clips.append(AudioFileClip(voice_path).volumex(1.0))
        if music_path and os.path.exists(music_path):
            audio_clips.append(
                AudioFileClip(music_path).volumex(0.35).set_duration(clip.duration)
            )
        if audio_clips:
            clip = clip.set_audio(CompositeAudioClip(audio_clips))
        clip.write_videofile(out_path, codec='libx264', audio_codec='aac',
                             logger=None, verbose=False)
        clip.close()
        return out_path
    except Exception as e:
        log.warning(f'moviepy mix : {e}')
        return video_path

# ── Génération vidéo via fallback (SD 1.5 multi-frames) ───────────────
def generate_video_fallback(prompt, negative_prompt, width, height,
                             num_frames, num_steps, guidance, fps, out_path):
    """Génère une vidéo en créant plusieurs frames avec SD 1.5."""
    import imageio
    from PIL import Image

    frames = []
    seeds  = list(range(num_frames))
    for i, seed in enumerate(seeds):
        generator = torch.Generator(device='cpu').manual_seed(seed)
        result = pipe_image(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=min(width, 512),
            height=min(height, 512),
            num_inference_steps=max(num_steps // 2, 10),
            guidance_scale=guidance,
            generator=generator,
        )
        frames.append(np.array(result.images[0]))
    imageio.mimsave(out_path, frames, fps=fps, codec='libx264')
    return out_path

# ── Worker génération ─────────────────────────────────────────────────
def run_generation(jid, data):
    try:
        prompt     = data.get('prompt', '')
        gen_type   = data.get('type', 'video')
        voice_on   = data.get('voice', True)
        music_on   = data.get('music', True)
        lang       = data.get('voice_lang', 'fr')
        mstyle     = data.get('music_style', 'cinematic')
        duration   = int(data.get('duration', 10))
        resolution = data.get('resolution', '1024x576')
        fps        = int(data.get('fps', 24))
        frames     = int(data.get('frames', 16))
        w, h       = [int(x) for x in resolution.split('x')]

        set_job(jid, status='processing', progress=5, step='Initialisation…')

        # ── IMAGE ─────────────────────────────────────────────────────
        if gen_type == 'image':
            set_job(jid, progress=20, step='Génération image SD 1.5…')
            if pipe_image is None:
                raise RuntimeError('Modèle image non chargé (cellule 3 non exécutée)')
            result = pipe_image(
                prompt=prompt,
                negative_prompt='low quality, blurry, distorted, watermark',
                height=min(h, 512), width=min(w, 512),
                num_inference_steps=30,
                guidance_scale=7.5,
            )
            img = result.images[0]
            # Upscale PIL vers résolution demandée
            from PIL import Image
            img = img.resize((w, h), Image.LANCZOS)
            out = os.path.join(OUTPUTS_DIR, f'BENYJOE_IMG_{jid}.png')
            img.save(out, 'PNG')
            set_job(jid, status='done', progress=100, step='Image prête !',
                    result=f'/outputs/{os.path.basename(out)}')
            return

        # ── ANIMATION (image → vidéo) ──────────────────────────────────
        if gen_type == 'animation':
            set_job(jid, progress=15, step='Génération image source…')
            if pipe_image is None:
                raise RuntimeError('Modèle image non chargé')
            result = pipe_image(
                prompt=prompt,
                negative_prompt='low quality, blurry',
                height=512, width=512,
                num_inference_steps=25,
                guidance_scale=7.5,
            )
            base_img = result.images[0]
            set_job(jid, progress=40, step='Animation frames…')
            import imageio
            from PIL import Image
            anim_frames = []
            base_np = np.array(base_img)
            for i in range(min(frames, 24)):
                # Zoom progressif simple
                scale = 1.0 + i * 0.005
                new_w = int(512 * scale)
                new_h = int(512 * scale)
                cropped = base_img.resize((new_w, new_h), Image.LANCZOS)
                # Recadrer au centre
                left = (new_w - 512) // 2
                top  = (new_h - 512) // 2
                frame = cropped.crop((left, top, left + 512, top + 512))
                frame = frame.resize((w, h), Image.LANCZOS)
                anim_frames.append(np.array(frame))
            raw_mp4 = os.path.join(OUTPUTS_DIR, f'anim_{jid}.mp4')
            imageio.mimsave(raw_mp4, anim_frames, fps=fps, codec='libx264')
            set_job(jid, progress=80, step='Finalisation…')
            final_mp4 = os.path.join(OUTPUTS_DIR, f'BENYJOE_FINAL_{jid}.mp4')
            voice_path = generate_voice(prompt, lang=lang) if voice_on else None
            music_desc = {'cinematic': 'epic cinematic score', 'electronic': 'electronic ambient'}.get(mstyle, 'cinematic music')
            music_path = generate_music(music_desc, duration=duration) if music_on else None
            if voice_path or music_path:
                mix_audio_with_video(raw_mp4, voice_path, music_path, final_mp4)
            else:
                import shutil; shutil.copy2(raw_mp4, final_mp4)
            set_job(jid, status='done', progress=100, step='Animation prête ✅',
                    result=f'/outputs/{os.path.basename(final_mp4)}')
            return

        # ── VIDÉO ─────────────────────────────────────────────────────
        set_job(jid, progress=10, step='Génération vidéo…')
        if pipe_video is None:
            raise RuntimeError('Modèle vidéo non chargé (cellule 3 non exécutée)')

        num_frames_  = min(max(frames, 8), 64)
        raw_mp4      = os.path.join(OUTPUTS_DIR, f'raw_{jid}.mp4')

        if ANIMATEDIFF:
            set_job(jid, progress=15, step='AnimateDiff frames…')
            output = pipe_video(
                prompt=prompt,
                negative_prompt='low quality, blurry, distorted',
                height=min(h, 512),
                width=min(w, 512),
                num_frames=num_frames_,
                num_inference_steps=20,
                guidance_scale=7.5,
            )
            from diffusers.utils import export_to_video
            export_to_video(output.frames[0], raw_mp4, fps=fps)
        else:
            set_job(jid, progress=15, step='Génération frames SD 1.5…')
            generate_video_fallback(
                prompt, 'low quality, blurry', w, h,
                num_frames_, 20, 7.5, fps, raw_mp4
            )

        set_job(jid, progress=60, step='Upscale HD…')
        hd_mp4 = os.path.join(OUTPUTS_DIR, f'hd_{jid}.mp4')
        import subprocess as sp
        r = sp.run([
            'ffmpeg', '-y', '-i', raw_mp4,
            '-vf', f'scale={w}:{h}:flags=lanczos',
            '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
            hd_mp4
        ], capture_output=True)
        if not os.path.exists(hd_mp4) or os.path.getsize(hd_mp4) == 0:
            hd_mp4 = raw_mp4

        voice_path = None
        if voice_on:
            set_job(jid, progress=75, step='Génération voix OFF…')
            voice_path = generate_voice(prompt, lang=lang)

        music_path = None
        if music_on:
            set_job(jid, progress=82, step='Composition MusicGen…')
            music_desc = {
                'cinematic':  'epic cinematic orchestral score',
                'electronic': 'electronic ambient synth music',
                'ambient':    'calm ambient meditation music',
                'epic':       'epic battle orchestral music',
                'oriental':   'oriental arabic music oud',
            }.get(mstyle, 'cinematic music')
            music_path = generate_music(music_desc, duration=max(duration, 10))

        final_mp4 = os.path.join(OUTPUTS_DIR, f'BENYJOE_FINAL_{jid}.mp4')
        if voice_path or music_path:
            set_job(jid, progress=90, step='Mixage audio + vidéo…')
            mix_audio_with_video(hd_mp4, voice_path, music_path, final_mp4)
        else:
            import shutil
            shutil.copy2(hd_mp4, final_mp4)

        for f in [raw_mp4, hd_mp4, voice_path, music_path]:
            if f and f != final_mp4 and os.path.exists(f):
                try: os.remove(f)
                except: pass

        set_job(jid, status='done', progress=100, step='Vidéo prête ✅',
                result=f'/outputs/{os.path.basename(final_mp4)}')
        print(f'✅ Vidéo prête : {os.path.basename(final_mp4)}')

    except Exception as e:
        log.error(f'Génération {jid} : {e}')
        import traceback; traceback.print_exc()
        set_job(jid, status='error', error=str(e))

# ── Flask app ─────────────────────────────────────────────────────────
flask_app = Flask('BENYJOE_IA')
CORS(flask_app)

@flask_app.route('/health')
def health():
    return jsonify({
        'status':   'ok',
        'platform': 'BENY-JOE IA',
        'founder':  'KHEDIM BENYAKHLEF dit BENY-JOE',
        'device':   DEVICE_NAME,
        'jobs':     len(jobs),
        'models': {
            'video':       pipe_video is not None,
            'animatediff': ANIMATEDIFF,
            'image':       pipe_image is not None,
            'music':       music_model is not None,
        }
    })

@flask_app.route('/generate', methods=['POST'])
def generate():
    data   = request.get_json(silent=True) or {}
    prompt = (data.get('prompt') or '').strip()
    if not prompt:
        return jsonify({'error': 'Prompt requis'}), 400
    jid = data.get('job_id') or str(uuid.uuid4())[:12]
    set_job(jid, status='queued', step='En file…', progress=0)
    t = threading.Thread(target=run_generation, args=(jid, data), daemon=True)
    t.start()
    return jsonify({'job_id': jid, 'status': 'queued'})

@flask_app.route('/api/jobs')
def api_jobs():
    with lock:
        return jsonify({'queue_size': len(jobs), 'jobs': dict(jobs)})

@flask_app.route('/api/jobs/<jid>')
def api_job(jid):
    with lock:
        job = jobs.get(jid)
    if not job:
        return jsonify({'error': 'Job introuvable'}), 404
    return jsonify(job)

@flask_app.route('/outputs/<path:filename>')
def serve_output(filename):
    fpath = os.path.join(OUTPUTS_DIR, filename)
    if not os.path.exists(fpath):
        abort(404)
    return send_file(fpath)

# ── Lancer Flask en thread ─────────────────────────────────────────────
import threading as _th
_flask_thread = _th.Thread(
    target=lambda: flask_app.run(
        host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False
    ),
    daemon=True, name='FlaskServer'
)
_flask_thread.start()
time.sleep(2)
print(f'✅ Serveur Flask actif sur http://localhost:{FLASK_PORT}')

# ── Tunnel ngrok ──────────────────────────────────────────────────────
from pyngrok import ngrok, conf
import requests as req

NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')
NGROK_URL   = ''

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    try: ngrok.kill()
    except: pass
    time.sleep(1)
    tunnel    = ngrok.connect(FLASK_PORT, 'http')
    NGROK_URL = tunnel.public_url
    os.environ['NGROK_URL'] = NGROK_URL
    print(f'✅ Tunnel ngrok : {NGROK_URL}')

    # Envoyer l'URL du tunnel à Render
    try:
        r = req.post(f'{RENDER_URL}/api/kaggle-url', json={
            'url':    NGROK_URL,
            'secret': SECRET_KEY,
            'device': DEVICE_NAME,
        }, timeout=12)
        st = '✅' if r.status_code in (200, 201) else f'⚠️  (code {r.status_code})'
        print(f'  📤 URL tunnel envoyée à Render : {st}')
    except Exception as e:
        print(f'  ⚠️  Render non joignable : {e}')
        print(f'      URL ngrok disponible localement : {NGROK_URL}')
else:
    print('⚠️  NGROK_TOKEN absent — tunnel non créé')
    print('  → Ajoutez NGROK_TOKEN dans Kaggle → Settings → Secrets')

print()
print('╔══════════════════════════════════════════════════════╗')
print('║        BENY-JOE IA — Cellule 4 opérationnelle !     ║')
print(f'║  Flask  : http://localhost:{FLASK_PORT}                    ║')
print(f'║  Render : {RENDER_URL[:44]:<44} ║')
print(f'║  ngrok  : {(NGROK_URL or "non disponible")[:44]:<44} ║')
print('╚══════════════════════════════════════════════════════╝')
print()
print('👉 Passez à la Cellule 5')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 4B — GÉNÉRATEUR LONGUE DURÉE ✅ JAX/TPU NATIF            ║
# ║  Chaque frame générée par JAX sur TPU (~2-5s par frame)            ║
# ║  N frames → concaténation → vidéo longue durée                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, gc, time, uuid, math, subprocess, shutil
import numpy as np
from pathlib import Path

OUTPUTS_DIR = '/kaggle/working/outputs'
CLIPS_DIR   = '/kaggle/working/clips'
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(CLIPS_DIR,   exist_ok=True)

# ── Constantes ────────────────────────────────────────────────────────
FRAMES_PER_CLIP = 16      # frames par clip
CLIP_FPS        = 8       # FPS de sortie
CLIP_DUR        = FRAMES_PER_CLIP / CLIP_FPS   # ~2s par clip
MAX_TOTAL_MIN   = 30      # max 30 minutes

print('╔══════════════════════════════════════════════════════╗')
print('║  Générateur Longue Durée — JAX/TPU                 ║')
mode = "JAX/TPU (~2-5s/frame)" if JAX_MODE else "CPU (~5-10min/frame)"
print(f'║  Mode : {mode:<42}║')
print(f'║  Clip : {FRAMES_PER_CLIP} frames × {CLIP_FPS}fps = {CLIP_DUR:.1f}s/clip{"":>27}║')
print('╚══════════════════════════════════════════════════════╝')
print()


# ── Générer 1 clip = FRAMES_PER_CLIP images concaténées ──────────────
def generate_clip_jax(prompt, negative_prompt, width, height,
                       n_frames, fps, out_path, base_seed=0):
    """
    Génère n_frames images via JAX/TPU et les assemble en MP4.
    Sur TPU : chaque appel JAX génère N_DEVICES images à la fois !
    """
    import imageio
    from PIL import Image

    all_frames = []
    calls_needed = math.ceil(n_frames / N_DEVICES)

    for call_i in range(calls_needed):
        seed = base_seed + call_i * 1000
        try:
            # JAX génère N_DEVICES images en parallèle (~2-5s)
            imgs = generate_image(
                prompt=prompt,
                negative_prompt=negative_prompt,
                height=min(height, 512),
                width=min(width, 512),
                steps=20,
                guidance=7.5,
                seed=seed,
            )
            for img in imgs:
                if len(all_frames) < n_frames:
                    # Upscale PIL vers résolution cible
                    if img.size != (width, height):
                        img = img.resize((width, height), Image.LANCZOS)
                    all_frames.append(np.array(img))
        except Exception as e:
            print(f'    ⚠️  Frame batch {call_i} : {e}')
            continue

    if not all_frames:
        return False

    # Sauvegarder en MP4
    try:
        imageio.mimsave(out_path, all_frames[:n_frames], fps=fps,
                        codec='libx264', quality=8)
        return True
    except Exception as e:
        print(f'    ⚠️  imageio save : {e}')
        try:
            imageio.mimsave(out_path, all_frames[:n_frames], fps=fps)
            return True
        except:
            return False


# ── Concaténation clips → vidéo longue ───────────────────────────────
def concatenate_clips(clip_paths, out_path):
    if not clip_paths:
        return False
    if len(clip_paths) == 1:
        shutil.copy2(clip_paths[0], out_path)
        return True

    list_path = out_path.replace('.mp4', '_list.txt')
    with open(list_path, 'w') as f:
        for cp in clip_paths:
            f.write(f"file '{cp}'\n")

    r = subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', list_path, '-c', 'copy', out_path
    ], capture_output=True, timeout=300)

    try: os.remove(list_path)
    except: pass

    if not (os.path.exists(out_path) and os.path.getsize(out_path) > 0):
        # fallback re-encode
        with open(list_path, 'w') as f:
            for cp in clip_paths:
                f.write(f"file '{cp}'\n")
        subprocess.run([
            'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
            '-i', list_path, '-c:v', 'libx264', '-crf', '20',
            '-preset', 'fast', '-pix_fmt', 'yuv420p', out_path
        ], capture_output=True, timeout=600)
        try: os.remove(list_path)
        except: pass

    return os.path.exists(out_path) and os.path.getsize(out_path) > 0


# ── Générateur principal ──────────────────────────────────────────────
def run_long_generation(jid, data):
    try:
        prompt          = data.get('prompt', 'cinematic landscape')
        neg_prompt      = data.get('negative_prompt',
                          'low quality, blurry, distorted, watermark')
        resolution      = data.get('resolution', '1024x576')
        total_duration  = min(int(data.get('duration', 10)), MAX_TOTAL_MIN * 60)
        fps             = int(data.get('fps', CLIP_FPS))
        voice_on        = data.get('voice', True)
        music_on        = data.get('music', True)
        lang            = data.get('voice_lang', 'fr')
        mstyle          = data.get('music_style', 'cinematic')
        base_seed       = int(data.get('seed', 42))
        w, h            = [int(x) for x in resolution.split('x')]

        n_clips = max(1, math.ceil(total_duration / CLIP_DUR))

        # Temps estimé selon mode
        if JAX_MODE:
            secs_per_clip = (FRAMES_PER_CLIP / N_DEVICES) * 4   # ~4s/batch JAX
        else:
            secs_per_clip = FRAMES_PER_CLIP * 300               # ~5min/frame CPU

        eta_min = (n_clips * secs_per_clip) / 60

        set_job(jid, status='processing', progress=2,
                step=f'{n_clips} clips × {CLIP_DUR:.1f}s — ETA ~{eta_min:.0f} min')

        print(f'\n🎬 [{jid}] Génération longue durée ({mode})')
        print(f'   Durée : {total_duration}s → {n_clips} clips')
        print(f'   ETA   : ~{eta_min:.0f} minutes')
        print()

        clip_paths = []
        clip_dir   = os.path.join(CLIPS_DIR, jid)
        os.makedirs(clip_dir, exist_ok=True)

        for i in range(n_clips):
            pct = 5 + int((i / n_clips) * 65)
            set_job(jid, progress=pct,
                    step=f'Clip {i+1}/{n_clips} ({pct}%) — {"JAX/TPU" if JAX_MODE else "CPU"}')

            clip_path = os.path.join(clip_dir, f'clip_{i:04d}.mp4')
            t0 = time.time()

            ok = generate_clip_jax(
                prompt=prompt,
                negative_prompt=neg_prompt,
                width=w, height=h,
                n_frames=FRAMES_PER_CLIP,
                fps=fps,
                out_path=clip_path,
                base_seed=base_seed + i * 100,
            )

            elapsed = time.time() - t0
            if ok and os.path.exists(clip_path):
                clip_paths.append(clip_path)
                remaining = (n_clips - i - 1) * elapsed
                print(f'  ✅ Clip {i+1:03d} — {elapsed:.0f}s — restant ~{remaining/60:.1f}min')
            else:
                print(f'  ⚠️  Clip {i+1:03d} échoué')

        if not clip_paths:
            raise RuntimeError('Aucun clip généré')

        # Concaténation
        set_job(jid, progress=72, step=f'Assemblage {len(clip_paths)} clips…')
        concat_path = os.path.join(OUTPUTS_DIR, f'concat_{jid}.mp4')
        if not concatenate_clips(clip_paths, concat_path):
            raise RuntimeError('Concaténation échouée')
        print(f'\n🔗 {len(clip_paths)} clips assemblés — {os.path.getsize(concat_path)//1024} KB')

        # Voix OFF
        voice_path = None
        if voice_on:
            set_job(jid, progress=78, step='Voix OFF gTTS…')
            voice_path = generate_voice(prompt, lang=lang)

        # Musique
        music_path = None
        if music_on and music_model is not None:
            set_job(jid, progress=84, step='MusicGen…')
            music_map = {
                'cinematic': 'epic cinematic orchestral',
                'electronic': 'electronic ambient synth',
                'ambient': 'calm ambient meditation',
                'epic': 'epic battle orchestral',
                'oriental': 'oriental arabic oud music',
                'jazz': 'smooth jazz piano bass',
                'nature': 'nature sounds birds forest',
            }
            music_path = generate_music(
                music_map.get(mstyle, 'cinematic music'),
                duration=min(total_duration, 30)
            )
            # Boucler si vidéo > 30s
            if music_path and total_duration > 30:
                loop_path = music_path.replace('.wav', '_loop.wav')
                subprocess.run([
                    'ffmpeg', '-y', '-stream_loop', '-1',
                    '-i', music_path, '-t', str(total_duration),
                    '-c', 'copy', loop_path
                ], capture_output=True, timeout=120)
                if os.path.exists(loop_path):
                    music_path = loop_path

        # Mixage final
        final_path = os.path.join(OUTPUTS_DIR, f'BENYJOE_FINAL_{jid}.mp4')
        set_job(jid, progress=90, step='Mixage final…')

        if voice_path or music_path:
            mix_audio_with_video(concat_path, voice_path, music_path, final_path)
        else:
            shutil.copy2(concat_path, final_path)

        # Nettoyage
        for f in [concat_path, voice_path, music_path]:
            if f and f != final_path and os.path.exists(f):
                try: os.remove(f)
                except: pass
        try: shutil.rmtree(clip_dir)
        except: pass

        sz = os.path.getsize(final_path) // (1024*1024)
        set_job(jid, status='done', progress=100,
                step=f'✅ Vidéo prête ({sz} MB)',
                result=f'/outputs/{os.path.basename(final_path)}')
        print(f'\n✅ [{jid}] TERMINÉ — {os.path.basename(final_path)} ({sz} MB)')

    except Exception as e:
        import traceback; traceback.print_exc()
        set_job(jid, status='error', error=str(e))

# ── Surcharge run_generation ──────────────────────────────────────────
_old_run_gen = run_generation

def run_generation(jid, data):
    if data.get('type', 'video') == 'video':
        run_long_generation(jid, data)
    else:
        _old_run_gen(jid, data)

# ── Stats ─────────────────────────────────────────────────────────────
print('✅ Générateur longue durée JAX/TPU actif')
print()
print('📊 Vitesses estimées :')
if JAX_MODE:
    print(f'   10s  → ~{math.ceil(10/CLIP_DUR)} clips → ~{math.ceil(10/CLIP_DUR)*int((FRAMES_PER_CLIP/N_DEVICES)*4)//60}min')
    print(f'   1min → ~{math.ceil(60/CLIP_DUR)} clips → ~{math.ceil(60/CLIP_DUR)*int((FRAMES_PER_CLIP/N_DEVICES)*4)//60}min')
    print(f'   5min → ~{math.ceil(300/CLIP_DUR)} clips → ~{math.ceil(300/CLIP_DUR)*int((FRAMES_PER_CLIP/N_DEVICES)*4)//60}min')
else:
    print('   Mode CPU actif — génération lente')
print()
print('👉 Passez à la Cellule 5')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Watcher vidéos : auto-push GitHub → Render            ║
# ║  ✅ Thread daemon — ne bloque PAS les autres cellules              ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, time, threading, requests
from datetime import datetime, timezone

RENDER_URL   = os.environ.get('RENDER_URL',     'https://benyjoe-ia.onrender.com')
SECRET_KEY   = os.environ.get('BENYJOE_SECRET', 'benyjoe-secret-2025')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN',   '')
GITHUB_REPO  = os.environ.get('GITHUB_REPO',    '')
GITHUB_API   = 'https://api.github.com'
OUTPUTS_DIR  = '/kaggle/working/outputs'
NGROK_URL    = os.environ.get('NGROK_URL',      '')

def get_ngrok_url():
    try:
        r = requests.get('http://localhost:4040/api/tunnels', timeout=5)
        for t in r.json().get('tunnels', []):
            if t.get('proto') == 'https':
                return t['public_url']
    except Exception:
        pass
    return os.environ.get('NGROK_URL', None)

def push_url_to_render(url):
    try:
        r = requests.post(f'{RENDER_URL}/api/kaggle-url', json={
            'url':    url,
            'secret': SECRET_KEY,
            'device': os.environ.get('DEVICE_NAME', 'Kaggle-TPU-v5e'),
        }, timeout=12)
        return r.status_code in (200, 201)
    except Exception as e:
        print(f'  ⚠️  Render push : {e}')
        return False

def upload_video_to_github(video_path, job_id=None):
    """Upload la vidéo dans un GitHub Release et retourne l'URL de téléchargement."""
    if not GITHUB_TOKEN or not GITHUB_REPO:
        print('  ⚠️  GITHUB_TOKEN ou GITHUB_REPO absent — upload impossible')
        return None
    if not os.path.exists(video_path) or os.path.getsize(video_path) == 0:
        print('  ⚠️  Fichier vidéo introuvable ou vide')
        return None

    headers  = {'Authorization': f'token {GITHUB_TOKEN}',
                'Accept':        'application/vnd.github.v3+json'}
    api_base = f'{GITHUB_API}/repos/{GITHUB_REPO}'
    tag      = 'benyjoe-ia-videos'

    try:
        # Récupérer ou créer le release
        r = requests.get(f'{api_base}/releases/tags/{tag}', headers=headers, timeout=10)
        if r.status_code == 200:
            upload_url = r.json()['upload_url'].split('{')[0]
        else:
            r2 = requests.post(f'{api_base}/releases', headers=headers, timeout=15,
                               json={'tag_name': tag, 'name': 'BENY-JOE IA Videos',
                                     'body': 'Vidéos générées par BENY-JOE IA', 'draft': False})
            if r2.status_code != 201:
                print(f'  ❌ Création release : {r2.status_code} — {r2.text[:200]}')
                return None
            upload_url = r2.json()['upload_url'].split('{')[0]

        # Upload l'asset
        ts         = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
        asset_name = f'{ts}_{os.path.basename(video_path)}'
        with open(video_path, 'rb') as fh:
            video_bytes = fh.read()
        r3 = requests.post(
            f'{upload_url}?name={asset_name}',
            headers={'Authorization': f'token {GITHUB_TOKEN}',
                     'Content-Type': 'video/mp4'},
            data=video_bytes, timeout=300
        )
        if r3.status_code == 201:
            return r3.json()['browser_download_url']
        else:
            print(f'  ❌ Upload asset : {r3.status_code}')
    except Exception as e:
        print(f'  ⚠️  GitHub upload : {e}')
    return None

def notify_render(video_url, job_id=None):
    try:
        requests.post(f'{RENDER_URL}/api/video-ready', json={
            'job_id':    job_id or 'unknown',
            'video_url': video_url,
            'source':    'kaggle-watcher',
            'device':    os.environ.get('DEVICE_NAME', 'Kaggle-TPU-v5e'),
            'secret':    SECRET_KEY,
        }, timeout=12)
    except: pass

# ── Watcher daemon ────────────────────────────────────────────────────
_watched     = set()
_watcher_run = True

def auto_push_watcher(interval=15):
    while _watcher_run:
        try:
            if os.path.isdir(OUTPUTS_DIR):
                for fname in list(os.listdir(OUTPUTS_DIR)):
                    # Surveille uniquement les fichiers finaux
                    if not fname.startswith('BENYJOE_FINAL_') or not fname.endswith('.mp4'):
                        continue
                    if fname in _watched:
                        continue
                    fpath = os.path.join(OUTPUTS_DIR, fname)
                    # Vérifier que le fichier est stable (terminé)
                    try:    sz1 = os.path.getsize(fpath)
                    except: continue
                    time.sleep(3)
                    try:    sz2 = os.path.getsize(fpath)
                    except: continue
                    if sz1 != sz2 or sz2 == 0:
                        continue

                    _watched.add(fname)
                    jid = fname.replace('BENYJOE_FINAL_', '').replace('.mp4', '')
                    print(f'\n🎬 [{datetime.now().strftime("%H:%M:%S")}] Vidéo détectée : {fname} ({sz2 // 1024} KB)')

                    # 1. Essayer upload GitHub
                    dl_url = upload_video_to_github(fpath, jid)
                    if dl_url:
                        notify_render(dl_url, job_id=jid)
                        print(f'  ✅ Vidéo envoyée à Render via GitHub : {dl_url}')
                    else:
                        # 2. Fallback : URL ngrok directe
                        ngrok_base = get_ngrok_url() or ''
                        if ngrok_base:
                            direct_url = f'{ngrok_base}/outputs/{fname}'
                            notify_render(direct_url, job_id=jid)
                            print(f'  📤 Notifié Render (ngrok fallback) : {direct_url}')
                        else:
                            print('  ⚠️  Impossible de notifier Render (pas de GitHub ni ngrok)')

        except Exception as e:
            print(f'Watcher erreur : {e}')
        time.sleep(interval)

# ── Démarrage ─────────────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════╗')
print('║   BENY-JOE IA — Cellule 5 : Watcher (arrière-plan)  ║')
print('╚══════════════════════════════════════════════════════╝')
print()

# Re-push URL ngrok
ngrok_url = get_ngrok_url()
render_ok  = False
if ngrok_url:
    print(f'  ✅ URL ngrok active : {ngrok_url}')
    render_ok = push_url_to_render(ngrok_url)
    print(f'  📤 Render push    : {"✅ OK" if render_ok else "⚠️  Échec"}')
else:
    print('  ⚠️  Pas de tunnel ngrok actif — relancez la Cellule 4')

print(f'  GITHUB_REPO : {GITHUB_REPO if GITHUB_REPO else "⚠️  non configuré"}')

# Démarrer le watcher en thread
watcher_thread = threading.Thread(
    target=auto_push_watcher, args=(15,), daemon=True, name='BenyJoeWatcher'
)
watcher_thread.start()

print()
print(f'  👁️  Watcher   : ✅ actif en arrière-plan (toutes les 15s)')
print(f'  📂 Surveille  : {OUTPUTS_DIR} (fichiers BENYJOE_FINAL_*.mp4)')
print()
print('✅ Cellule 5 terminée — passez à la Cellule 6 (keep-alive)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — KEEP-ALIVE PERMANENT ♾️                               ║
# ║  ⚠️  LANCER EN DERNIER — cette cellule tourne indéfiniment        ║
# ║  Elle maintient le notebook actif (anti-timeout Kaggle)            ║
# ║  Statut affiché toutes les 60 secondes                             ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, time, requests
from datetime import datetime

RENDER_URL  = os.environ.get('RENDER_URL',     'https://benyjoe-ia.onrender.com')
SECRET_KEY  = os.environ.get('BENYJOE_SECRET', 'benyjoe-secret-2025')
OUTPUTS_DIR = '/kaggle/working/outputs'

def get_ngrok_url():
    try:
        r = requests.get('http://localhost:4040/api/tunnels', timeout=5)
        for t in r.json().get('tunnels', []):
            if t.get('proto') == 'https':
                return t['public_url']
    except Exception:
        pass
    return os.environ.get('NGROK_URL', 'non disponible')

def push_url_to_render(url):
    try:
        r = requests.post(f'{RENDER_URL}/api/kaggle-url', json={
            'url':    url,
            'secret': SECRET_KEY,
            'device': os.environ.get('DEVICE_NAME', 'Kaggle-TPU-v5e'),
        }, timeout=12)
        return r.status_code in (200, 201)
    except Exception:
        return False

print('╔══════════════════════════════════════════════════════╗')
print('║      BENY-JOE IA — KEEP-ALIVE PERMANENT ♾️           ║')
print('║  Le notebook reste actif — statut toutes les 60s    ║')
print('║  Fondé par KHEDIM BENYAKHLEF dit BENY-JOE           ║')
print('╚══════════════════════════════════════════════════════╝')
print()

# Vérification initiale
ngrok_live = get_ngrok_url()
nb_videos  = len([f for f in os.listdir(OUTPUTS_DIR) if f.endswith('.mp4')]) if os.path.isdir(OUTPUTS_DIR) else 0
print(f'  🌐 ngrok  : {ngrok_live}')
print(f'  📁 Outputs : {OUTPUTS_DIR} ({nb_videos} vidéo(s))')
print(f'  🖥️  Device  : {os.environ.get("DEVICE_NAME", "?")}')
print()
print('✅ Keep-alive démarré — CTRL+C ou Stop pour arrêter')
print()

cycle = 0
while True:
    time.sleep(60)
    cycle    += 1
    ts         = datetime.now().strftime('%H:%M:%S')
    ngrok_live = get_ngrok_url()
    nb_jobs    = len(jobs) if 'jobs' in dir() else '?'
    nb_videos  = len([f for f in os.listdir(OUTPUTS_DIR) if f.endswith('.mp4')]) if os.path.isdir(OUTPUTS_DIR) else 0

    status_icon = '💓' if cycle % 2 == 0 else '🔋'
    print(f'{status_icon} [{ts}] Keep-alive #{cycle:04d} | ngrok: {ngrok_live[:40]} | jobs: {nb_jobs} | vidéos: {nb_videos}')

    # Re-push URL Render toutes les 5 minutes (cycle de 5)
    if cycle % 5 == 0:
        if ngrok_live and ngrok_live != 'non disponible':
            ok = push_url_to_render(ngrok_live)
            print(f'   🔄 Re-push URL Render : {"✅" if ok else "⚠️"}')